# LiDAR exploration: building context

This notebook validates classifications, spatial coverage and ground-normalized heights before choosing a reconstruction method. It consumes a small derived crop; it never downloads or stores the complete CNIG tile.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import laspy
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from matplotlib.colors import to_hex
from scipy.spatial import cKDTree

from urbanstock3d.processors.lidar import points_in_polygon
from urbanstock3d.providers.pnoa_lidar import footprint_rings_utm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BUILDING_ID = 'ES.SDGC.BU.4531917YJ2743B'
CROP_PATH = PROJECT_ROOT / 'outputs' / BUILDING_ID / 'lidar_context_crop.laz'
BUILDING_PATH = PROJECT_ROOT / 'outputs' / BUILDING_ID / 'building.geojson'
assert CROP_PATH.exists(), f'Generate the crop first: {CROP_PATH}'
assert BUILDING_PATH.exists(), f'Missing building geometry: {BUILDING_PATH}'

CLASS_STYLES = {
    1: ('Unclassified', '#7f7f7f'),
    2: ('Ground', '#8c564b'),
    3: ('Low vegetation', '#bcbd22'),
    4: ('Medium vegetation', '#2ca02c'),
    5: ('High vegetation', '#006400'),
    6: ('Building', '#d62728'),
    7: ('Noise', '#9467bd'),
    12: ('Legacy overlap', '#17becf'),
}
DEFAULT_CLASS_STYLE = ('Other', '#000000')
HEIGHT_COLORMAP = 'viridis'

## 1. Load the reproducible context crop

In [ ]:
cloud = laspy.read(CROP_PATH)
x = np.asarray(cloud.x)
y = np.asarray(cloud.y)
z = np.asarray(cloud.z)
classification = np.asarray(cloud.classification, dtype=np.uint8)

print(f'Points: {len(cloud.points):,}')
print(f'CRS: {cloud.header.parse_crs()}')
print(f'Point format: {cloud.header.point_format.id}')
print(f'Dimensions: {list(cloud.point_format.dimension_names)}')
Counter(classification)

## 2. Inspect classifications in plan view

Class 12 is a legacy overlap classification in this tile, so it is visualized but excluded from geometric measurements.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
for class_id in np.unique(classification):
    mask = classification == class_id
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    ax.scatter(x[mask], y[mask], s=1, c=color, label=f'{label} ({mask.sum():,})')
ax.set(title='PNOA-LiDAR classifications', xlabel='Easting (m)', ylabel='Northing (m)')
ax.set_aspect('equal')
ax.legend(markerscale=5, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

## 3. Compare usable points and legacy overlap

In [ ]:
usable = classification != 12
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for class_id in np.unique(classification[usable]):
    mask = usable & (classification == class_id)
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    axes[0].scatter(x[mask], y[mask], c=color, s=2, label=label)
axes[0].set_title(f'Usable points ({usable.sum():,})')
axes[0].legend(markerscale=4)
overlap_label, overlap_color = CLASS_STYLES[12]
axes[1].scatter(x[~usable], y[~usable], c=overlap_color, s=2)
axes[1].set_title(f'{overlap_label} — class 12 ({(~usable).sum():,})')
for ax in axes:
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
plt.show()

## 4. Normalize elevation against nearby classified ground

In [ ]:
ground = classification == 2
ground_z = float(np.median(z[ground]))
height = z - ground_z
print(f'Ground reference (median class 2): {ground_z:.2f} m')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes[0].hist(height[usable], bins=80, color='#7f7f7f')
axes[0].set(title='Ground-normalized height distribution', xlabel='Height (m)', ylabel='Points')
for class_id in np.unique(classification[usable]):
    mask = usable & (classification == class_id)
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    axes[1].scatter(x[mask], height[mask], c=color, s=2, label=label)
axes[1].set(title='East-height cross-section', xlabel='Easting (m)', ylabel='Height (m)')
axes[1].legend(markerscale=4)
plt.show()

## 5. Isolate classified roof points inside the cadastral footprint

In [ ]:
building = json.loads(BUILDING_PATH.read_text(encoding='utf-8'))
rings = footprint_rings_utm(building['geometry']['coordinates'])
in_footprint = points_in_polygon(x, y, rings)
roof = in_footprint & (classification == 6)
roof_height = height[roof]

print(f'Classified roof points in footprint: {roof.sum():,}')
print(f'Roof height p50: {np.percentile(roof_height, 50):.2f} m')
print(f'Roof height p95: {np.percentile(roof_height, 95):.2f} m')

fig, ax = plt.subplots(figsize=(9, 7))
plot = ax.scatter(x[roof], y[roof], c=roof_height, s=8, cmap=HEIGHT_COLORMAP)
ax.set(title='Roof points inside cadastral footprint', xlabel='Easting (m)', ylabel='Northing (m)')
ax.set_aspect('equal')
fig.colorbar(plot, ax=ax, label='Height above ground (m)')
plt.show()

## 6. Explore the point cloud interactively in 3D

The cloud is deterministically downsampled when necessary to keep interaction responsive. Each classification uses the same categorical color as the 2D views, while the vertical axis shows height above the local ground reference. Use the legend to show or hide individual classes.

In [ ]:
# Limit the displayed points to keep rotation and zoom responsive.
max_points = 30_000
indices = np.flatnonzero(usable)

if len(indices) > max_points:
    rng = np.random.default_rng(42)
    indices = rng.choice(indices, max_points, replace=False)

figure = go.Figure()
for class_id in np.unique(classification[indices]):
    class_indices = indices[classification[indices] == class_id]
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    figure.add_trace(
        go.Scatter3d(
            x=x[class_indices],
            y=y[class_indices],
            z=height[class_indices],
            mode='markers',
            name=f'{label} — class {class_id}',
            marker={'size': 1.5, 'color': color, 'opacity': 0.8},
            customdata=np.full(len(class_indices), class_id),
            hovertemplate=(
                'Easting: %{x:.2f} m<br>'
                'Northing: %{y:.2f} m<br>'
                'Height: %{z:.2f} m<br>'
                'Class: %{customdata}<extra>%{fullData.name}</extra>'
            ),
        )
    )

figure.update_layout(
    title='Ground-normalized LiDAR point cloud',
    scene={
        'xaxis_title': 'Easting (m)',
        'yaxis_title': 'Northing (m)',
        'zaxis_title': 'Height above ground (m)',
        'aspectmode': 'data',
    },
    legend={'title': 'PNOA-LiDAR class'},
    height=750,
)

figure.show()

## 7. Estimate local roof geometry

A point normal and geometric descriptors are estimated from the covariance eigenvalues of each point's spherical neighborhood. Several radii are compared first because the radius controls the balance between noise sensitivity and loss of small roof structures. Only classified building points inside the cadastral footprint participate in this experiment.

In [ ]:
def compute_local_features(points, radius, min_neighbors=6):
    """Estimate covariance-based features for every sufficiently supported point."""
    tree = cKDTree(points)
    neighborhoods = tree.query_ball_point(points, radius)
    point_count = len(points)
    neighbor_count = np.fromiter(
        (len(neighbors) for neighbors in neighborhoods),
        dtype=np.int32,
        count=point_count,
    )
    normals = np.full((point_count, 3), np.nan)
    eigenvalues = np.full((point_count, 3), np.nan)

    for point_index, neighbor_indices in enumerate(neighborhoods):
        if len(neighbor_indices) < min_neighbors:
            continue
        neighbors = points[neighbor_indices]
        centered = neighbors - neighbors.mean(axis=0)
        covariance = centered.T @ centered / len(neighbors)
        values, vectors = np.linalg.eigh(covariance)
        values = np.maximum(values, 0.0)
        normal = vectors[:, 0]
        normals[point_index] = normal if normal[2] >= 0 else -normal
        eigenvalues[point_index] = values

    smallest, middle, largest = eigenvalues.T
    safe_largest = np.where(largest > 0, largest, np.nan)
    eigenvalue_sum = eigenvalues.sum(axis=1)
    return {
        'neighbor_count': neighbor_count,
        'valid': np.isfinite(largest),
        'normals': normals,
        'linearity': (largest - middle) / safe_largest,
        'planarity': (middle - smallest) / safe_largest,
        'scattering': smallest / safe_largest,
        'surface_variation': smallest / np.where(eigenvalue_sum > 0, eigenvalue_sum, np.nan),
    }


roof_points = np.column_stack((x[roof], y[roof], z[roof]))
candidate_radii = (0.75, 1.0, 1.5, 2.0, 2.5)
radius_comparison = {}

for radius in candidate_radii:
    candidate = compute_local_features(roof_points, radius)
    valid = candidate['valid']
    radius_comparison[radius] = candidate
    print(
        f'Radius {radius:.2f} m | valid {valid.mean():.1%} | '
        f'median neighbors {np.median(candidate["neighbor_count"]):.0f} | '
        f'median planarity {np.nanmedian(candidate["planarity"]):.3f}'
    )

### Working neighborhood

For this sample, a 1.5 m radius is the initial working choice: it gives broad point coverage while retaining substantially smaller neighborhoods than the 2.0–2.5 m alternatives. This is an experimental parameter, not yet a production default.

In [ ]:
WORKING_RADIUS_M = 1.5
local_features = radius_comparison[WORKING_RADIUS_M]
feature_valid = local_features['valid']
roof_x = roof_points[:, 0]
roof_y = roof_points[:, 1]
normal_z = np.clip(local_features['normals'][:, 2], 0.0, 1.0)
slope_degrees = np.degrees(np.arccos(normal_z))

fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
feature_views = (
    ('Planarity', local_features['planarity'], 'viridis'),
    ('Surface variation', local_features['surface_variation'], 'magma'),
    ('Estimated slope (degrees)', slope_degrees, 'cividis'),
    ('Neighborhood size', local_features['neighbor_count'], 'plasma'),
)

for ax, (title, values, colormap) in zip(axes.flat, feature_views, strict=True):
    plot = ax.scatter(
        roof_x[feature_valid],
        roof_y[feature_valid],
        c=values[feature_valid],
        s=12,
        cmap=colormap,
    )
    ax.set(title=title, xlabel='Easting (m)', ylabel='Northing (m)')
    ax.set_aspect('equal')
    fig.colorbar(plot, ax=ax)

plt.show()
print(f'Valid local features: {feature_valid.sum():,} / {len(feature_valid):,}')

## 8. Segment candidate roof planes with RANSAC

RANSAC repeatedly proposes a plane from three points and retains the model supported by the most points. The experiment uses only locally stable points, fits planes in centered coordinates for numerical stability, and measures orthogonal point-to-plane distance. The thresholds below are deliberately explicit and remain experimental.

In [ ]:
def plane_residuals(points, coefficients):
    """Calculate orthogonal distances to z = ax + by + c."""
    a, b, c = coefficients
    vertical_residual = points[:, 2] - (a * points[:, 0] + b * points[:, 1] + c)
    return np.abs(vertical_residual) / np.sqrt(a**2 + b**2 + 1.0)


def fit_plane(points):
    """Fit z = ax + by + c by least squares."""
    design = np.column_stack((points[:, 0], points[:, 1], np.ones(len(points))))
    coefficients, _, _, _ = np.linalg.lstsq(design, points[:, 2], rcond=None)
    return coefficients


def segment_roof_planes(
    points,
    eligible,
    *,
    distance_threshold=0.25,
    min_points=40,
    max_planes=8,
    iterations=750,
    seed=42,
):
    """Extract dominant non-vertical planes with sequential RANSAC."""
    rng = np.random.default_rng(seed)
    labels = np.full(len(points), -1, dtype=np.int16)
    remaining = np.flatnonzero(eligible)
    models = []

    for plane_id in range(max_planes):
        if len(remaining) < min_points:
            break
        best_inliers = np.array([], dtype=np.int64)

        for _ in range(iterations):
            sample_indices = rng.choice(remaining, size=3, replace=False)
            sample = points[sample_indices]
            design = np.column_stack((sample[:, 0], sample[:, 1], np.ones(3)))
            if np.linalg.matrix_rank(design) < 3:
                continue
            candidate = fit_plane(sample)
            residuals = plane_residuals(points[remaining], candidate)
            inliers = remaining[residuals <= distance_threshold]
            if len(inliers) > len(best_inliers):
                best_inliers = inliers

        if len(best_inliers) < min_points:
            break

        model = fit_plane(points[best_inliers])
        refined_residuals = plane_residuals(points[remaining], model)
        inliers = remaining[refined_residuals <= distance_threshold]
        if len(inliers) < min_points:
            break
        labels[inliers] = plane_id
        models.append(model)
        remaining = remaining[refined_residuals > distance_threshold]

    return labels, models


xy_origin = roof_points[:, :2].mean(axis=0)
roof_local = np.column_stack(
    (roof_points[:, 0] - xy_origin[0], roof_points[:, 1] - xy_origin[1], roof_height)
)
stable = (
    feature_valid
    & (local_features['planarity'] >= 0.30)
    & (local_features['surface_variation'] <= 0.10)
)
plane_labels, plane_models = segment_roof_planes(roof_local, stable)
assigned = plane_labels >= 0

print(f'Geometrically stable points: {stable.sum():,} / {len(stable):,}')
print(f'Points assigned to planes: {assigned.sum():,} / {len(stable):,}')
print(f'Candidate planes: {len(plane_models)}')
for plane_id, model in enumerate(plane_models):
    plane_mask = plane_labels == plane_id
    slope = np.degrees(np.arctan(np.hypot(model[0], model[1])))
    residual = np.median(plane_residuals(roof_local[plane_mask], model))
    print(
        f'Plane {plane_id + 1}: {plane_mask.sum():,} points | '
        f'slope {slope:.1f} degrees | median residual {residual:.3f} m'
    )

In [ ]:
plane_palette = plt.colormaps['tab10']
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(roof_x[~assigned], roof_y[~assigned], s=8, c='#bdbdbd', label='Unassigned')
for plane_id in range(len(plane_models)):
    mask = plane_labels == plane_id
    ax.scatter(
        roof_x[mask],
        roof_y[mask],
        s=12,
        color=plane_palette(plane_id),
        label=f'Plane {plane_id + 1}',
    )
ax.set(title='RANSAC roof-plane candidates', xlabel='Easting (m)', ylabel='Northing (m)')
ax.set_aspect('equal')
ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

plane_figure = go.Figure()
plane_figure.add_trace(
    go.Scatter3d(
        x=roof_x[~assigned],
        y=roof_y[~assigned],
        z=roof_height[~assigned],
        mode='markers',
        name='Unassigned',
        marker={'size': 2, 'color': '#bdbdbd', 'opacity': 0.5},
    )
)
for plane_id in range(len(plane_models)):
    mask = plane_labels == plane_id
    color = to_hex(plane_palette(plane_id))
    plane_figure.add_trace(
        go.Scatter3d(
            x=roof_x[mask],
            y=roof_y[mask],
            z=roof_height[mask],
            mode='markers',
            name=f'Plane {plane_id + 1}',
            marker={'size': 2.5, 'color': color, 'opacity': 0.85},
        )
    )
plane_figure.update_layout(
    title='Interactive roof-plane candidates',
    scene={
        'xaxis_title': 'Easting (m)',
        'yaxis_title': 'Northing (m)',
        'zaxis_title': 'Height above ground (m)',
        'aspectmode': 'data',
    },
    height=750,
)
plane_figure.show()

## Observations

Record conclusions here after running the cells:

- Is class 12 spatially redundant with the usable coverage?
- Are class-6 points clean inside the cadastral footprint?
- Does the ground median represent the local terrain adequately?
- Are multiple roof-height modes visible?
- Which artifacts should be removed before estimating roof planes?
- Does the 1.5 m neighborhood preserve visible roof boundaries?
- Which planarity and surface-variation ranges separate stable roof surfaces from edges?
- Do coplanar but disconnected roof regions need a spatial connectivity step?
- Are the 0.25 m distance and 40-point support thresholds appropriate for this density?